<a href="https://colab.research.google.com/github/Neloy-Barman/Fast-Efficient-LLM-Inference-with-vLLM/blob/main/LLM_Compression_Getting_Started.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Requirements Installation

In [ ]:
!pip install uv

In [ ]:
!uv pip install llmcompressor

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.1/48.1 MB 15.3 MB/s  0:00:03
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached ninja-1.13.2-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (5.4 kB)
  Using cached torch-2.14.0-cp313-cp313-manylinux_2_28_x86_64.whl.metadata (37 kB)
INFO: pip is looking at multiple versions of llmcompressor to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of llmcompressor to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.

# Google Drive Setup
Mounting Google Drive and prepare a directory named `llm_compressor_practice` to store models

In [ ]:
import os
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Pre-create the target directory
DRIVE_BASE_DIR = "/content/drive/MyDrive/llm_compressor_practice"
os.makedirs(DRIVE_BASE_DIR, exist_ok=True)
print(f"Google Drive folder ready at: {DRIVE_BASE_DIR}")

Mounted at /content/drive
Google Drive folder ready at: /content/drive/MyDrive/llm_compressor_practice


# Checking GPU

In [ ]:
!nvidia-smi

Thu Nov 19 12:05:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Model and Tokenizer Load from HuggingFace

## Defining the Model ID and prepare a dir to store the model

In [ ]:
# MODEL_ID = "Qwen/Qwen3-30B-A3B"
MODEL_ID = "Qwen/Qwen3-0.6B"
BASE_MODEL_SAVE_DIR = os.path.join(DRIVE_BASE_DIR, f"{MODEL_ID}")
os.makedirs(BASE_MODEL_SAVE_DIR, exist_ok=True)

## Load, Save and reload Model and tokenizer

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
# # Load Model
# model = AutoModelForCausalLM.from_pretrained(MODEL_ID)
# # Load Tokenizer
# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# # Save model and tokenizer to the defined directory.
# model.save_pretrained(BASE_MODEL_SAVE_DIR)
# tokenizer.save_pretrained(BASE_MODEL_SAVE_DIR)

# Load model and tokenizer from saved dir
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_SAVE_DIR)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_SAVE_DIR)
print(f"Successfully loaded base model and tokenizer from: {BASE_MODEL_SAVE_DIR}")

Loading weights:   0%|          | 0/310 [00:01<?, ?it/s]

Successfully loaded base model and tokenizer from: /content/drive/MyDrive/llm_compressor_practice/Qwen/Qwen3-0.6B


# LLM Compressor

## Recipe initialization with Quantization Modifier, Scheme and target layers

In [ ]:
from llmcompressor.modifiers.quantization import QuantizationModifier

# Configure the quantization algorithm and scheme.
# In this case, we:
#   * quantize the weights to FP8 using RTN with block_size 128
#   * quantize the activations dynamically to FP8 during inference
recipe = QuantizationModifier(
    targets="Linear",
    scheme="FP8_BLOCK",
    ignore=["lm_head", "re:.*mlp.gate$"],
)

## Quantization Implementation

In [ ]:
from llmcompressor import oneshot

oneshot(
    model=model,
    recipe=recipe
)

2026-11-19T06:58:27.7631 | __init__ | WARNING - Disabling tokenizer parallelism due to threading conflict between FastTokenizer and Datasets. Set TOKENIZERS_PARALLELISM=false to suppress this warning.
2026-11-19T06:58:29.0757 | reset | INFO - Compression lifecycle reset
2026-11-19T06:58:29.0823 | from_modifiers | INFO - Creating recipe from modifiers


Applying quantization config: 100%|██████████| 196/196 [00:00<00:00, 5139.50it/s]

2026-11-19T06:58:29.1579 | initialize | INFO - Compression lifecycle initialized for 1 modifiers
2026-11-19T06:58:29.1606 | IndependentPipeline | INFO - Inferred `DataFreePipeline` for `QuantizationModifier`


2026-11-19T06:58:33.0142 | finalize | INFO - Compression lifecycle finalized for 1 modifiers


Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layer

## Directory Definition after applying quantization

In [ ]:
# Define the local save directory name and target Drive path
SAVE_DIR = MODEL_ID.split("/")[1] + "-FP8-BLOCK"
DRIVE_DIR = os.path.join(DRIVE_BASE_DIR, SAVE_DIR)
os.makedirs(DRIVE_DIR, exist_ok=True)

## Save quantized model and tokenizer to directory

In [ ]:
import os
# Save model and tokenizer directly to Google Drive
model.save_pretrained(DRIVE_DIR)
tokenizer.save_pretrained(DRIVE_DIR)

print(f"Successfully saved the model and tokenizer to: {DRIVE_DIR}")

Compressing model: 100%|██████████| 196/196 [00:00<00:00, 208.62it/s]


2026-11-19T06:58:53.7749 | _retie_embeddings | INFO - Re-tied input/output embeddings; saving a single shared table.


/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:3430: UserWarning: Attempting to save a model with offloaded modules. Ensure that unallocated cpu memory exceeds the `shard_size` (50GB default)
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

2026-11-19T06:58:59.0757 | resave_config | INFO - Resaved original config with patched fields to /content/drive/MyDrive/llm_compressor_practice/Qwen3-0.6B-FP8-BLOCK/config.json


Dispatching model: 100%|██████████| 427/427 [00:00<00:00, 1680.86it/s]


Successfully saved the model and tokenizer to: /content/drive/MyDrive/llm_compressor_practice/Qwen3-0.6B-FP8-BLOCK


## Load model and tokenizer from saved directory

In [ ]:
# Load the base model from the saved Google Drive folder
model = AutoModelForCausalLM.from_pretrained(DRIVE_DIR)
# Load the tokenizer from the saved Google Drive folder
tokenizer = AutoTokenizer.from_pretrained(DRIVE_DIR)

ImportError: compressed-tensors>=0.15.0 is required for compressed-tensors quantization. Please install it with `pip install compressed-tensors>=0.15.0`.

## Dispatch compressed model and perform test

In [ ]:
from compressed_tensors.offload import dispatch_model

# Confirm generations of the quantized model look sane.
print("========== SAMPLE GENERATION ==============")

dispatch_model(model)

input_ids = tokenizer("Hello my name is", return_tensors="pt").input_ids.to(model.device)

output = model.generate(input_ids, max_new_tokens=20)

print(tokenizer.decode(output[0]))

print("==========================================")

========== SAMPLE GENERATION ==============


Decompressing model: 100%|██████████| 196/196 [00:00<00:00, 2544.53it/s]


Hello my name is johndoe. I'm a student from a school in the city of miami. I


In [ ]:
from vllm import LLM, SamplingParams

model = LLM(model=DRIVE_DIR)

sampling_params = SamplingParams(
    max_tokens=256,
)

In [ ]:
output = model.generate(
    "What is maching learning?",
    sampling_params=sampling_params,
)
print(output)

for output in outputs:
  print(output.outputs[0].text)

In [ ]:
vllm serve DRIVE_DIR